In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline

# 1. Loading data
loan_data = pd.read_csv(r"C:\Users\USER\credit_risk_dataset.csv")

# 2. Splitting raw features and target variable
y_raw = loan_data['loan_status']
X_raw = loan_data.drop(columns=['loan_status'])

X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.20, random_state=42, stratify=y_raw)

# 3. Isolating target proxies / leakage columns
train_loan_grade_val = X_train['loan_grade'].copy()
test_loan_grade_val = X_test['loan_grade'].copy()

X_train = X_train.drop(columns=['loan_grade'])
X_test = X_test.drop(columns=['loan_grade'])

# 4. Outlier Filtering (Including person_income to prevent mathematical errors in pipelines)
X_train_clean = X_train[
    (X_train['person_age'] <= 90) & 
    (X_train['person_emp_length'] <= 60) & 
    (X_train['person_income'] > 0)
].copy()
y_train = y_train.loc[X_train_clean.index]

X_test_clean = X_test[
    (X_test['person_age'] <= 90) & 
    (X_test['person_emp_length'] <= 60) & 
    (X_test['person_income'] > 0)
].copy()
y_test = y_test.loc[X_test_clean.index]

# 5. Extracting Medians from training distribution ONLY
int_rate_median = X_train_clean['loan_int_rate'].median()
emp_length_median = X_train_clean['person_emp_length'].median()

# Imputing Missing numerical entries across both subsets
X_train_clean['loan_int_rate'] = X_train_clean['loan_int_rate'].fillna(int_rate_median)
X_train_clean['person_emp_length'] = X_train_clean['person_emp_length'].fillna(emp_length_median)

X_test_clean['loan_int_rate'] = X_test_clean['loan_int_rate'].fillna(int_rate_median)
X_test_clean['person_emp_length'] = X_test_clean['person_emp_length'].fillna(emp_length_median)

# --- SECTION FOR LEGACY MANUAL MODEL EVALUATION ---
# (Maintained so your standalone metrics loops continue running)
categorical_cols = ['person_home_ownership', 'loan_intent', 'cb_person_default_on_file']

X_train_encoded = pd.get_dummies(X_train_clean, columns=categorical_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_clean, columns=categorical_cols, drop_first=True)

# Add missing columns manually created by drop_first logic alignment
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

# Explicit boolean casting to 1/0 integers
bool_cols_train = X_train_encoded.select_dtypes(include=['bool']).columns
X_train_encoded[bool_cols_train] = X_train_encoded[bool_cols_train].astype(int)

bool_cols_test = X_test_encoded.select_dtypes(include=['bool']).columns
X_test_encoded[bool_cols_test] = X_test_encoded[bool_cols_test].astype(int)

# Persist files to local folder tracking
split_output_dir = "./credit_risk_split_output"
os.makedirs(split_output_dir, exist_ok=True)
print("=== SAVING BLIND SPLIT DATA TO DISK ===")

X_train_encoded.to_csv(os.path.join(split_output_dir, "X_train_clean.csv"), index=False)
y_train.to_csv(os.path.join(split_output_dir, "y_train.csv"), index=False)
train_loan_grade_val.to_csv(os.path.join(split_output_dir, "train_loan_grade.csv"), index=False)

X_test_encoded.to_csv(os.path.join(split_output_dir, "X_test_clean.csv"), index=False)
y_test.to_csv(os.path.join(split_output_dir, "y_test.csv"), index=False)
test_loan_grade_val.to_csv(os.path.join(split_output_dir, "test_loan_grade.csv"), index=False)

print("=== INITIALIZING STRATIFIED CROSS-VALIDATION ===")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = RandomForestClassifier(
    n_estimators=150, 
    max_depth=10,
    min_samples_leaf=5, 
    class_weight='balanced', 
    random_state=42, 
    n_jobs=-1
)

X_train_matrix = X_train_encoded.values
y_train_vector = y_train.values

train_auc_scores, val_auc_scores, fold_accuracies = [], [], []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_matrix, y_train_vector), 1):
    X_tr, X_val = X_train_matrix[train_idx], X_train_matrix[val_idx]
    y_tr, y_val = y_train_vector[train_idx], y_train_vector[val_idx]
    
    model.fit(X_tr, y_tr)
    predictions = model.predict(X_val)
    prob_val = model.predict_proba(X_val)[:, 1]
    
    acc = accuracy_score(y_val, predictions)
    auc_val = roc_auc_score(y_val, prob_val)
    
    fold_accuracies.append(acc)
    val_auc_scores.append(auc_val)
    
    prob_train = model.predict_proba(X_tr)[:, 1]
    auc_train = roc_auc_score(y_tr, prob_train)
    train_auc_scores.append(auc_train)
    
    print(f"Fold {fold} -> Train AUC: {auc_train:.4f} | Val AUC: {auc_val:.4f} | Val Acc: {acc:.4f}")

# --- PRODUCTION EXPORTS FOR LEGACY ARTIFACTS ---
print("\n=== EXPORTING BASELINE PRODUCTION ARTIFACTS ===")
# FIXED: Re-fit legacy standalone baseline model and align export names
model.fit(X_train_encoded, y_train)
joblib.dump(model, r"C:\Users\USER\Loan Default Predictor\loan_risk_model.pkl")
joblib.dump(X_train_encoded.columns.tolist(), r"C:\Users\USER\Loan Default Predictor\model_columns.pkl")
print("✔️ 'loan_risk_model.pkl' and 'model_columns.pkl' successfully exported!")


# --- SECTION FOR BUILDING INDESTRUCTIBLE INDUSTRIAL PIPELINE ---
print("\n=== BUILDING INDESTRUCTIBLE PRODUCTION PIPELINE ===")

numeric_cols = ['person_age', 'person_emp_length', 'loan_amnt', 'loan_int_rate']
categorical_cols = ['person_home_ownership', 'loan_intent', 'cb_person_default_on_file']

# Functional Transformation steps mapped inside scikit-learn boundaries
def log_transform_income(df):
    return np.log(df[['person_income']].astype(float))

def compute_dti_ratio(df):
    dti = df['loan_amnt'].astype(float) / df['person_income'].astype(float)
    return pd.DataFrame(dti, columns=['loan_percent_income'])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_cols),
        ('log_inc', FunctionTransformer(log_transform_income, validate=False), ['person_income']),
        ('dti_calc', FunctionTransformer(compute_dti_ratio, validate=False), ['loan_amnt', 'person_income'])
    ]
)

production_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=150, 
        max_depth=10,
        min_samples_leaf=5, 
        class_weight='balanced', 
        random_state=42, 
        n_jobs=-1
    ))
])

# Fit the entire preprocessing array and classifier simultaneously on raw features
production_pipeline.fit(X_train_clean, y_train)

# Export the immutable artifact directly to your server framework deployment path
joblib.dump(production_pipeline, r"C:\Users\USER\Loan Default Predictor\loan_risk_pipeline.pkl")
print("✔️ Integrated production pipeline 'loan_risk_pipeline.pkl' saved successfully!")

=== SAVING BLIND SPLIT DATA TO DISK ===

--- PHYSICAL DISK VERIFICATION ---
📁 Saved: test_loan_grade.csv       | Size: 19.10 KB
📁 Saved: train_loan_grade.csv      | Size: 76.37 KB
📁 Saved: X_test_clean.csv          | Size: 313.03 KB
📁 Saved: X_train_clean.csv         | Size: 1244.76 KB
📁 Saved: y_test.csv                | Size: 18.65 KB
📁 Saved: y_train.csv               | Size: 74.19 KB
=== INITIALIZING STRATIFIED CROSS-VALIDATION ===
Fold 1 -> Train AUC: 0.9432 | Val AUC: 0.9267 | Val Acc: 0.8993
Fold 2 -> Train AUC: 0.9444 | Val AUC: 0.9244 | Val Acc: 0.8967
Fold 3 -> Train AUC: 0.9457 | Val AUC: 0.9124 | Val Acc: 0.8870
Fold 4 -> Train AUC: 0.9460 | Val AUC: 0.9115 | Val Acc: 0.8872
Fold 5 -> Train AUC: 0.9474 | Val AUC: 0.9048 | Val Acc: 0.8908

--- DIAGNOSTIC SUMMARY RESUME ---
Average Training ROC-AUC:   0.9454
Average Validation ROC-AUC: 0.9160
The Overfitting Gap:        0.0294

=== FINAL REVENUE-PROTECTION TEST EVALUATION ===
Final Vault Test Set ROC-AUC: 0.9208

--- CONFUSIO

: 